In [1]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from dotenv import load_dotenv
import os
import requests
import json
groq_api_key = os.getenv("GROQ_API_KEY")



load_dotenv()

API_KEY = os.getenv("EXCHANGE_RATE_API_KEY")

# -----------------------------
# LLM
# -----------------------------

llm = ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)
# -----------------------------
# Tool
# -----------------------------

@tool
def currency_converter(from_currency: str,
                       to_currency: str,
                       amount: float):
    """
    Convert one currency into another using live exchange rates.
    """

    url = f"https://v6.exchangerate-api.com/v6/{API_KEY}/pair/{from_currency}/{to_currency}/{amount}"

    response = requests.get(url)

    data = response.json()

    if data["result"] == "success":
        return f"{amount} {from_currency} = {data['conversion_result']} {to_currency}"

    return "Currency conversion failed."


# -----------------------------
# Bind Tool
# -----------------------------

llm_with_tools = llm.bind_tools([currency_converter])

# -----------------------------
# User Question
# -----------------------------

messages = [
    HumanMessage(
        content="what is the conversion factor between USD and PKR, and based on that can you convert 10 usd into pkr"
    )
]

# -----------------------------
# First LLM Call
# -----------------------------

ai_message = llm_with_tools.invoke(messages)

messages.append(ai_message)

# -----------------------------
# Execute Tool
# -----------------------------

for tool_call in ai_message.tool_calls:

    if tool_call["name"] == "currency_converter":

        tool_result = currency_converter.invoke(tool_call)

        messages.append(

            ToolMessage(
                content=tool_result,
                tool_call_id=tool_call["id"]
            )

        )

# -----------------------------
# Final LLM Call
# -----------------------------

final_response = llm_with_tools.invoke(messages)

print(final_response.content)

The current conversion rate is:

**1 USD = 278.0684 PKR**

So, converting **10 USD** gives:

**10 USD × 278.0684 PKR/USD = 2,780.684 PKR**

Therefore, 10 USD is approximately **2,780.68 PKR** (rounded to two decimal places).
